# Build a streaming chatbot with indah (Colab)

This notebook loads a small instruct model with **transformers** and streams its
reply, token by token, into a chat UI built with
[**indah**](https://github.com/leejianrong/indah) - all in one Colab runtime, with
no Node and no separate server.

It runs on **free Colab**: a sub-1B model is snappy on a T4 GPU and fine on CPU.
For a GPU, pick *Runtime -> Change runtime type -> T4 GPU* (optional).

The design keeps two things apart on purpose (ADR-0009):

- **the model layer** is plain Python with no indah imports - it just yields
  tokens, so a real app drops its own model in behind the same shape;
- **the indah layer** wires that stream of tokens into a live page.

## 1. Install

`indah` is pure-Python (the frontend ships pre-built in the wheel, so there is no
Node step). `transformers`, `accelerate`, and `torch` are *your* dependencies -
they are not indah's, which is what keeps indah light. Colab already has `torch`.

In [ ]:
!pip install --quiet --pre indah transformers accelerate

## 2. The model layer (plain Python, no indah)

`chat_stream` yields the assistant's reply token by token. transformers'
`TextIteratorStreamer` runs `model.generate` on a background thread; we pull each
token off it with `asyncio.to_thread`, so the asyncio event loop stays free and
indah's UI never freezes while the model generates.

The default is **Qwen2.5-0.5B-Instruct** (small, fast, Apache-2.0). Swap in
`unsloth/Llama-3.2-1B-Instruct` (or any instruct model) by changing `MODEL`.

In [ ]:
import asyncio
import threading

from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # try: "unsloth/Llama-3.2-1B-Instruct"

print(f"Loading {MODEL} (the first run downloads weights)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto", device_map="auto")


async def chat_stream(messages, *, max_new_tokens=512):
    """Yield an assistant reply token by token for a chat `messages` list.

    No indah imports: this is the portable half of the app. Generation runs on a
    background thread; each token is pulled off with asyncio.to_thread so the event
    loop is free between tokens and the UI stays live.
    """
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    def generate():
        try:
            model.generate(
                **inputs,
                streamer=streamer,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
            )
        finally:
            streamer.end()  # unblock the consumer even if generate() raised

    threading.Thread(target=generate, daemon=True).start()

    done = object()
    while True:
        token = await asyncio.to_thread(next, streamer, done)
        if token is done:
            break
        yield token

## 3. The indah app

The whole conversation lives in one `StreamText`: each turn is fed in as *append*
deltas - the user's line, then the assistant's tokens as they arrive. The
transcript grows at O(token) cost on the wire and re-renders correctly if the
proxy drops and the browser reconnects.

We batch a few tokens per SSE frame (`COALESCE_TOKENS`) to keep the wire light on
Colab's proxy; it stays visibly a stream.

In [ ]:
import indah
from indah import Button, Column, Session, Signal, StreamText, Text, TextInput

COALESCE_TOKENS = 3


async def coalesced(stream, every=COALESCE_TOKENS):
    buffer = []
    async for token in stream:
        buffer.append(token)
        if len(buffer) >= every:
            yield "".join(buffer)
            buffer = []
    if buffer:
        yield "".join(buffer)


prompt = Signal("")
status = Signal("Ask me something.")
busy = Signal(False)
transcript = StreamText(label="Conversation")
messages = []  # plain [{"role", "content"}] chat history


async def on_send():
    question = prompt.value.strip()
    if not question or busy.value:
        return
    busy.set(True)
    status.set("Generating...")
    prompt.set("")
    messages.append({"role": "user", "content": question})
    transcript.feed(f"You: {question}\n")
    transcript.feed("Assistant: ")

    parts = []
    async for chunk in coalesced(chat_stream(messages)):
        transcript.feed(chunk)
        parts.append(chunk)

    transcript.feed("\n\n")
    messages.append({"role": "assistant", "content": "".join(parts)})
    busy.set(False)
    status.set("Ask me something.")


page = Column(
    children=[
        Text("indah chatbot: a small local LLM, streaming into the page"),
        TextInput(prompt, placeholder="Type a message, then click Send", label="Message"),
        Button("Send", on_click=on_send),
        Text(status),
        transcript,
    ]
)

handle = indah.launch(indah.create_app(session=Session(page)))

## 4. Try it

The app is embedded above. Type a message and click **Send**: the reply streams in
token by token. Send another and the transcript keeps the earlier turns - the
model sees the whole history each time.

**How it stays responsive:** `on_send` is an `async` handler that indah runs as a
background task, and generation itself runs on a worker thread, so the page never
blocks while the model works.

**Graduating this app:** `chat_stream` never imports indah. When an engineering
team picks up the prototype, that function lifts straight into their own backend
unchanged (ADR-0009); only the thin indah layer is indah-specific.

When you are done, stop the server:

In [ ]:
handle.stop()